# RSNA 2026 Deep Learning Lab · Vibe Coding 101: Building Radiology Applications Without Programming Experience
**Talk 1 · Prompt Engineering Fundamentals: How to Communicate Effectively with AI Models**
Tugba Akinci D'Antonoli, MD · University Hospital Basel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RSNA/AI-Deep-Learning-Lab-2026/blob/main/05-Vibe-Coding-101/Lecture1-Prompt-Engineering.ipynb)

**One notebook for the whole workshop:** follow along during the talks (parts 1–4), then continue in the hands-on session (part 5).
To keep your own changes: **File → Save a copy in Drive** (you need a Google account).

This notebook follows the lecture step by step:

1. **How LLMs work**: tokens, next-token prediction and temperature
2. **Prompt engineering**: from a vague prompt to a good one, Kim et al.'s six levers, zero-/few-shot, chain-of-thought, self-consistency, reverse prompting
3. **Vibe coding**: let the model write a small radiology tool, then **validate** it
4. **Safeguards** for clinician-built tools
5. **Hands-on**: build and iterate on your own tool

> ⚠️ **All reports in this notebook are synthetic.** Never paste identifiable patient data into a general-purpose LLM.

**How to use it:** run the cells from top to bottom (Shift + Enter). Cells marked ✍️ are exercises: change the prompt and run again.

## 0 · Setup

Choose **one** way to reach a model:

| Option | What you need |
|---|---|
| `gemini` | A free API key from [Google AI Studio](https://aistudio.google.com/apikey), saved in Colab **Secrets** (🔑 icon on the left) as `GOOGLE_API_KEY` |
| `colab` | Colab's built-in AI (`google.colab.ai`), no key needed; availability depends on your Colab plan |
| `openai` | An API key saved in Secrets as `OPENAI_API_KEY` |
| `anthropic` | An API key saved in Secrets as `ANTHROPIC_API_KEY` |
| `offline` | No model at all: canned answers so you can still walk through the notebook (e.g., if the Wi-Fi fails) |

Model names change often. Leave `MODEL` empty to use the default, or type a current model name.

In [ ]:
#@title Choose your model provider
PROVIDER = "gemini"  #@param ["gemini", "colab", "openai", "anthropic", "offline"]
MODEL = ""  #@param {type:"string"}

DEFAULT_MODELS = {
    "gemini": "gemini-2.5-flash",
    "colab": "",                      # empty = Colab's default model
    "openai": "gpt-5-mini",
    "anthropic": "claude-haiku-5-5",
    "offline": "offline-demo",
}
PACKAGES = {"gemini": "google-genai", "openai": "openai", "anthropic": "anthropic"}

import subprocess, sys
if PROVIDER in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGES[PROVIDER]], check=False)
print(f"Provider: {PROVIDER} · model: {MODEL or DEFAULT_MODELS[PROVIDER] or '(Colab default)'}")

In [ ]:
#@title Helper functions (run once; no need to read)
import os, re, json, textwrap
from collections import Counter
from IPython.display import display, Markdown, HTML

def _key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_clients = {}

def ask(prompt, system=None, temperature=0.2, max_tokens=4000):
    """Send one prompt to the chosen model and return the answer as text."""
    model = MODEL or DEFAULT_MODELS[PROVIDER]
    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types
        client = _clients.setdefault("gemini", genai.Client(api_key=_key("GOOGLE_API_KEY")))
        cfg = types.GenerateContentConfig(system_instruction=system, temperature=temperature,
                                          max_output_tokens=max_tokens)
        return client.models.generate_content(model=model, contents=prompt, config=cfg).text or ""
    if PROVIDER == "colab":
        from google.colab import ai
        full = (f"{system}\n\n" if system else "") + prompt
        return ai.generate_text(full, model_name=model) if model else ai.generate_text(full)
    if PROVIDER == "openai":
        from openai import OpenAI
        client = _clients.setdefault("openai", OpenAI(api_key=_key("OPENAI_API_KEY")))
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        try:
            r = client.chat.completions.create(model=model, messages=msgs, temperature=temperature)
        except Exception:   # some models only accept the default temperature
            r = client.chat.completions.create(model=model, messages=msgs)
        return r.choices[0].message.content
    if PROVIDER == "anthropic":
        import anthropic
        client = _clients.setdefault("anthropic", anthropic.Anthropic(api_key=_key("ANTHROPIC_API_KEY")))
        kw = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "user", "content": prompt}])
        if system: kw["system"] = system
        r = client.messages.create(**kw)
        return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
    return _offline(prompt)

def show(text, title=None):
    """Pretty-print a model answer."""
    if title: display(Markdown(f"#### {title}"))
    display(Markdown(text if text else "_(empty answer)_"))

def extract_block(text, lang=""):
    """Return the first ```lang ... ``` code block in a model answer (or the whole text)."""
    m = re.search(r"```(?:" + lang + r")?\s*\n(.*?)```", text, re.S | re.I)
    return (m.group(1) if m else text).strip()

def parse_json(text):
    """Parse JSON from a model answer, tolerating code fences and extra prose."""
    t = extract_block(text, "json")
    m = re.search(r"[\[{].*[\]}]", t, re.S)
    return json.loads(m.group(0) if m else t)

# ---------- offline demo answers (used only when PROVIDER = "offline") ----------
def _offline(prompt):
    p = prompt.lower()
    if "def prostate_volume_psad" in p or "python function" in p:
        return """```python
import math

def prostate_volume_psad(length_cm, width_cm, height_cm, psa_ng_ml):
    # Ellipsoid prostate volume (mL) and PSA density (ng/mL/mL)
    for name, v in [("length_cm", length_cm), ("width_cm", width_cm),
                    ("height_cm", height_cm), ("psa_ng_ml", psa_ng_ml)]:
        if not isinstance(v, (int, float)) or v <= 0:
            raise ValueError(f"{name} must be a positive number")
    volume = length_cm * width_cm * height_cm * math.pi / 6
    return {"volume_ml": round(volume, 1), "psa_density": round(psa_ng_ml / volume, 3)}
```"""
    if "html" in p:
        return """```html
<!DOCTYPE html><html><head><meta charset="utf-8"><title>PSA density</title>
<style>body{font-family:sans-serif;max-width:420px;margin:20px}input{width:80px}</style></head>
<body><h3>Prostate volume &amp; PSA density (offline demo)</h3>
<p>L <input id="l" type="number" step="0.1"> W <input id="w" type="number" step="0.1"> H <input id="h" type="number" step="0.1"> cm</p>
<p>PSA <input id="p" type="number" step="0.1"> ng/mL</p><button onclick="calc()">Calculate</button><p id="out"></p>
<script>function calc(){const l=+document.getElementById('l').value,w=+document.getElementById('w').value,
h=+document.getElementById('h').value,p=+document.getElementById('p').value;
if([l,w,h,p].some(v=>!(v>0))){document.getElementById('out').textContent='Enter positive numbers.';return;}
const v=l*w*h*Math.PI/6;document.getElementById('out').textContent=
'Volume '+v.toFixed(1)+' mL · PSA density '+(p/v).toFixed(3)+' ng/mL/mL';}</script></body></html>
```"""
    if "json" in p:
        rows = []
        for rid in re.findall(r"report (r\d)", p):
            rows.append({"id": rid.upper(), **GROUND_TRUTH[rid.upper()]})
        return "```json\n" + json.dumps(rows if len(rows) != 1 else rows[0], indent=2) + "\n```"
    if "yes or no" in p:
        return "YES"
    return "_[offline demo] A real model answer would appear here. Switch PROVIDER to try it live._"

print("Helpers ready. Testing the connection...")
print(ask("Reply with exactly: Connection OK"))

### Synthetic reports used throughout
Four short, invented brain MRI reports, plus the "ground truth" we will use to **check** the model.

In [ ]:
REPORTS = {
 "R1": "MRI brain with and without contrast. Findings: There is a 2.3 cm enhancing mass in the left frontal lobe "
       "with surrounding vasogenic edema and 4 mm rightward midline shift. No other lesions. "
       "Impression: Solitary enhancing left frontal mass, suspicious for high-grade glioma or metastasis.",
 "R2": "MRI brain with and without contrast. Findings: Two non-enhancing T2/FLAIR hyperintense lesions in the "
       "periventricular white matter, one on each side; the larger measures 0.8 cm on the right. "
       "Impression: Nonspecific white matter lesions; demyelinating disease is a consideration.",
 "R3": "MRI brain with and without contrast. Findings: Three enhancing lesions: right cerebellar hemisphere 1.5 cm, "
       "left parietal lobe 0.9 cm and left occipital lobe 0.6 cm, each with mild surrounding edema. "
       "Impression: Multiple enhancing lesions, most consistent with metastases.",
 "R4": "MRI brain without contrast. Findings: No intracranial mass, hemorrhage or acute infarct. Ventricles and sulci "
       "are normal for age. Impression: Normal MRI of the brain.",
}
GROUND_TRUTH = {
 "R1": {"n_lesions": 1, "largest_cm": 2.3, "enhancing": True,  "side": "left"},
 "R2": {"n_lesions": 2, "largest_cm": 0.8, "enhancing": False, "side": "bilateral"},
 "R3": {"n_lesions": 3, "largest_cm": 1.5, "enhancing": True,  "side": "bilateral"},
 "R4": {"n_lesions": 0, "largest_cm": None, "enhancing": False, "side": "none"},
}
for k, v in REPORTS.items(): print(f"{k}: {v}\n")

## 1 · Under the hood: how LLMs work, simply

An LLM reads text as **tokens** (word pieces) and predicts the **next token**, again and again.
**Temperature** controls how adventurous that choice is: low = predictable, high = more varied.

In [ ]:
#@title 1.1 · See the tokens
sentence = "Ring-enhancing lesion in the left periventricular white matter."
try:
    import tiktoken
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=False)
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    ids = enc.encode(sentence)
    print(f"{len(ids)} tokens:")
    print(" | ".join(enc.decode([i]) for i in ids))
except Exception as e:
    print("Tokenizer not available offline; rough word-piece split instead:")
    print(" | ".join(re.findall(r"\w{1,6}|\W", sentence)))

In [ ]:
#@title 1.2 · Temperature: same prompt, three runs each
prompt = "Complete with ONE short phrase only: 'The most common primary brain tumor in adults is ...'"
for t in (0.0, 1.0):
    print(f"--- temperature {t} ---")
    for _ in range(3):
        print("  ", ask(prompt, temperature=t).strip()[:120])

✍️ **Try it:** ask for a creative title for a radiology talk at temperature 0.0 and 1.0. Which would you use for report writing?

## 2 · Prompt engineering

### 2.1 · Vague versus specific
The same task, asked two ways. The second prompt states a **goal**, the **input**, the **audience** and the **output format**.

In [ ]:
vague = "Summarize this: " + REPORTS["R1"]
specific = f"""You are a neuroradiologist writing for a referring neurosurgeon.
Goal: summarize the MRI report below in exactly 3 bullet points: key finding, mass effect, differential diagnosis.
Use only information in the report. Do not add recommendations.

<report>
{REPORTS["R1"]}
</report>"""
show(ask(vague), "Vague prompt")
show(ask(specific), "Specific prompt")

### 2.2 · Kim et al.'s six levers
Kim et al. (RadioGraphics 2025) list six ways to improve any prompt: **context, clear instructions, role, tone and format, constraints and delimiters, and examples (shot-prompting)**.
The builder below assembles them for you. ✍️ Change any field and run again.

In [ ]:
def build_prompt(context="", instruction="", role="", tone_format="", constraints="", examples="", text=""):
    parts = []
    if role:        parts.append(f"Role: {role}")
    if context:     parts.append(f"Context: {context}")
    if instruction: parts.append(f"Task: {instruction}")
    if tone_format: parts.append(f"Tone and format: {tone_format}")
    if constraints: parts.append(f"Constraints: {constraints}")
    if examples:    parts.append(f"Examples:\n{examples}")
    if text:        parts.append(f"<input>\n{text}\n</input>")
    return "\n\n".join(parts)

p = build_prompt(
    role="Experienced neuroradiologist",
    context="The report goes to a general practitioner who is not familiar with neuroradiology terms.",
    instruction="Rewrite the impression so a general practitioner understands the key message and urgency.",
    tone_format="Plain clinical English, maximum 60 words.",
    constraints="Use only facts from the input between the <input> tags. Do not invent findings.",
    text=REPORTS["R3"],
)
print(p, "\n" + "-"*60)
show(ask(p))

### 2.3 · Zero-shot versus few-shot: structured extraction, then **check** the result
We ask for JSON so that the answer can be compared with the ground truth automatically. This is the habit that matters most: **validate, don't trust.**

In [ ]:
SCHEMA = '{"id": str, "n_lesions": int, "largest_cm": float or null, "enhancing": true/false, "side": "left"|"right"|"bilateral"|"none"}'

def extraction_prompt(rid, few_shot=False):
    example = ""
    if few_shot:
        example = ('Example\nReport: "Single 1.1 cm non-enhancing lesion in the right temporal lobe."\n'
                   'Output: {"id": "X", "n_lesions": 1, "largest_cm": 1.1, "enhancing": false, "side": "right"}\n\n')
    return (f"Extract the following fields from the MRI report and answer with JSON only, using this schema: {SCHEMA}.\n"
            f"'side' is 'bilateral' if lesions are on both sides.\n\n{example}"
            f"Report {rid.lower()}: \"{REPORTS[rid]}\"")

def score(few_shot):
    correct = total = 0
    for rid in REPORTS:
        try:
            out = parse_json(ask(extraction_prompt(rid, few_shot), temperature=0))
        except Exception as e:
            print(rid, "could not parse JSON:", e); total += len(GROUND_TRUTH[rid]); continue
        for field, truth in GROUND_TRUTH[rid].items():
            ok = out.get(field) == truth
            correct += ok; total += 1
            if not ok: print(f"  {rid}.{field}: model={out.get(field)!r}  truth={truth!r}")
    print(f"{'Few-shot' if few_shot else 'Zero-shot'} accuracy: {correct}/{total} fields")

score(few_shot=False)
score(few_shot=True)

✍️ **Try it:** add a fifth report with a tricky phrase (for example *"no enhancing lesions"*) and its ground truth. Does the model still get it right?

### 2.4 · Chain-of-thought: find the errors in a report
This synthetic report contains **two deliberate errors** (inspired by LLM error-detection studies). Asking the model to reason step by step, and to compare findings with the impression, usually helps.

In [ ]:
ERROR_REPORT = ("CT head without contrast. Findings: Hypodensity in the left MCA territory involving the insula and "
                "frontal operculum, measuring 3.1 cm, consistent with an acute infarct. No hemorrhage. "
                "Impression: Acute right MCA territory infarct measuring 1.3 cm. No hemorrhage.")
cot = f"""You are checking a radiology report for errors before it is signed.
Think step by step:
1. List each finding in the Findings section (location, side, size).
2. List each statement in the Impression.
3. Compare them and list every inconsistency (side, size, location, missing or extra findings).
Finish with a line 'ERRORS FOUND: <number>'.

<report>
{ERROR_REPORT}
</report>"""
show(ask(cot, temperature=0))
print("Expected: 2 errors (side: left vs right; size: 3.1 cm vs 1.3 cm).")

### 2.5 · Self-consistency: ask several times, take the majority
For yes/no decisions, sampling the answer several times and voting makes the result more stable.

In [ ]:
critical_report = ("CT head: Acute left holohemispheric subdural hematoma, 12 mm thick, with 6 mm rightward "
                   "midline shift and effacement of the left lateral ventricle.")
q = ("Does this report contain a critical finding that requires immediate phone communication with the "
     f"referring physician? Answer YES or NO only.\n\n<report>\n{critical_report}\n</report>")
votes = [ask(q, temperature=1.0).strip().upper()[:3] for _ in range(5)]
print("Votes:", votes)
print("Majority:", Counter(votes).most_common(1)[0][0])

### 2.6 · Patient-friendly language and translation
Constraints matter: reading level, length, no new information, and a pointer back to the doctor.

In [ ]:
LANGUAGE = "German"  #@param ["English", "German", "French", "Spanish", "Italian", "Turkish"]
p = build_prompt(
    role="Radiologist explaining results to a patient",
    instruction=f"Explain this MRI report to the patient in {LANGUAGE}.",
    tone_format="Warm and calm, reading level of a 12-year-old, maximum 120 words.",
    constraints=("Use only facts from the report. Do not give a diagnosis that is not in the report. "
                 "End by saying that the treating doctor will discuss the next steps."),
    text=REPORTS["R2"],
)
show(ask(p))

### 2.7 · Reverse prompt engineering
Have an output you like? Ask the model which prompt would produce it, then reuse that prompt.

In [ ]:
target_output = """TECHNIQUE: MRI brain with and without contrast.
FINDINGS:
- Lesions: 3 enhancing lesions (largest 1.5 cm, right cerebellum).
- Edema / mass effect: mild perilesional edema, no midline shift.
- Other: no hemorrhage, ventricles normal.
IMPRESSION: Multiple enhancing lesions, most consistent with metastases."""
show(ask("Write the prompt that would make an LLM turn a free-text brain MRI report into exactly this structured "
         f"format. Make the prompt reusable for any report.\n\n<desired_output>\n{target_output}\n</desired_output>"))

## 3 · Vibe coding: from a sentence to a tool

Describe the tool in plain language; the model writes the code. Then **you** do what the model cannot: review it and test it against values you know.

### 3.1 · A Python function, with deterministic tests

In [ ]:
spec = """Write a Python function prostate_volume_psad(length_cm, width_cm, height_cm, psa_ng_ml).
- Prostate volume uses the ellipsoid formula: length x width x height x pi/6, in mL.
- PSA density = PSA / volume.
- Raise ValueError if any input is not a positive number.
- Return a dict {"volume_ml": volume rounded to 1 decimal, "psa_density": rounded to 3 decimals}.
- Standard library only. Return only the code in one ```python block."""
generated = extract_block(ask(spec, temperature=0), "python")
print(generated)

In [ ]:
#@title Review the code above, then tick the box and run this cell
REVIEWED = False  #@param {type:"boolean"}
if not REVIEWED:
    print("Read the generated code first, then set REVIEWED = True and run again.")
else:
    ns = {}
    exec(generated, ns)                       # run the generated code in its own namespace
    f = ns["prostate_volume_psad"]
    tests = [((4.5, 3.2, 4.0, 6.0), 30.2, 0.199),    # values worked out by hand
             ((5.0, 4.0, 4.0, 10.0), 41.9, 0.239)]
    for args, vol, dens in tests:
        r = f(*args)
        ok = abs(r["volume_ml"] - vol) < 0.15 and abs(r["psa_density"] - dens) < 0.002
        print(("PASS" if ok else "FAIL"), args, "->", r)
    try:
        f(-1, 3, 4, 5); print("FAIL: negative length accepted")
    except ValueError:
        print("PASS: negative length rejected")

### 3.2 · A single-file, offline web tool (safety by design)
Like the PI-RADS tool in Hwang et al. (Korean J Radiol 2026): one HTML file, no network, no patient identifiers.
We then run a quick automatic **safety scan** before opening it.

In [ ]:
html_spec = """Create a single self-contained HTML file for a prostate volume and PSA density calculator.
Requirements:
- Inputs: length, width, height (cm) and PSA (ng/mL). No name, date of birth or any patient identifier fields.
- Ellipsoid formula (L x W x H x pi/6); show volume in mL and PSA density with 3 decimals.
- Validate inputs (positive numbers only) and show a clear message otherwise.
- No external libraries, fonts or images; no network requests; no storage of data.
- Clean, minimal design. Return only the code in one ```html block."""
html_tool = extract_block(ask(html_spec, temperature=0, max_tokens=6000), "html")

RED_FLAGS = ["http://", "https://", "fetch(", "XMLHttpRequest", "WebSocket", "<script src", "localStorage",
             "sessionStorage", "document.cookie", "navigator.sendBeacon"]
found = [f for f in RED_FLAGS if f.lower() in html_tool.lower()]
print("Safety scan:", "no red flags found" if not found else f"REVIEW NEEDED -> {found}")
print(f"{len(html_tool.splitlines())} lines of HTML generated")

In [ ]:
#@title Open the tool (only after the safety scan) and download it
if not found:
    display(HTML(html_tool))
    with open("psad_calculator.html", "w") as fh: fh.write(html_tool)
    try:
        from google.colab import files; files.download("psad_calculator.html")
    except Exception:
        print("Saved as psad_calculator.html")
else:
    print("Fix the flagged items first (ask the model to remove them), then run again.")

✍️ **Your turn:** change `html_spec` to build your own small tool, for example
- a **contrast safety checklist** (eGFR band → action),
- a **teaching quiz** with five neuroradiology signs,
- a **Fazekas score** helper.

Then: test it with values you know, and keep the safety scan.

## 4 · Safeguards for vibe-coded tools (condensed from Hwang et al., Korean J Radiol 2026)

| Threat | Safeguard |
|---|---|
| Insecure generated code | Prefer offline, single-file, dependency-free tools. *Running is not proof of safety.* |
| Hallucinated or fake packages | Avoid third-party dependencies; verify any package before installing |
| Leaked secrets | Keep keys out of code (use Colab Secrets, as in this notebook) |
| Patient data in prompts, screenshots, logs | Never paste identifiable data into a general-purpose LLM; use synthetic or de-identified data |
| Excessive data collection | Collect no identifier the task does not need; prefer tools that store nothing |
| Ungoverned deployment | Institutional privacy and security review (and IRB for research) before use with patient data |

## Take home
- An LLM predicts the next token; **how you prompt** shapes what it predicts.
- Use the six levers: context, instructions, role, format, constraints, examples.
- Vibe coding lets you build your own tools: **start small, keep patient data out, validate everything.**

## References
- Kim et al. Optimizing large language models in radiology and mitigating pitfalls: prompt engineering and fine-tuning. *RadioGraphics* 2025;45(4):e240073.
- Dietrich et al. Prompt engineering for large language models in interventional radiology. *AJR* 2025;225:e2532956.
- Nguyen et al. *Clin Imaging* 2024;115:110276.
- Bera et al. *J Am Coll Radiol* 2026;23:1283–1288.
- Hwang et al. *Korean J Radiol* 2026;27:929–935.
- Lavista Ferres et al. *J Am Coll Radiol* 2026. DOI: 10.1016/j.jacr.2026.06.008
- Nedoszytko M. Vibe Coding versus Clinical Reality. *NEJM AI* 2026. DOI: 10.1056/AIp2600342
- A. Karpathy (@karpathy) on X: January 2023, February 2025, January 2026.

## 5 · Hands-on: build your own tool
This part is for the hands-on session. The loop is always the same: **describe → generate → safety scan → test → refine.**

Fill in the six building blocks of your tool below. Keep it small, offline, and free of patient data.

In [ ]:
#@title 5.1 · Describe your tool
GOAL        = "A teaching quiz with 5 classic neuroradiology signs (name, description, one-line explanation)"  #@param {type:"string"}
USERS       = "Radiology residents"  #@param {type:"string"}
INPUTS      = "The user picks an answer from 4 options per question"  #@param {type:"string"}
OUTPUTS     = "Score at the end and the correct answer with a short explanation after each question"  #@param {type:"string"}
LAYOUT      = "One question per screen, large buttons, clean minimal design"  #@param {type:"string"}
CONSTRAINTS = "Single self-contained HTML file; no external libraries, fonts or images; no network requests; no storage; no patient data fields"  #@param {type:"string"}

my_spec = f"""Create a single self-contained HTML file.
Goal: {GOAL}
Users: {USERS}
Inputs: {INPUTS}
Outputs: {OUTPUTS}
Layout: {LAYOUT}
Constraints: {CONSTRAINTS}
Return only the code in one ```html block."""
my_tool = extract_block(ask(my_spec, temperature=0, max_tokens=8000), "html")
found = [f for f in RED_FLAGS if f.lower() in my_tool.lower()]
print("Safety scan:", "no red flags found" if not found else f"REVIEW NEEDED -> {found}")
print(f"{len(my_tool.splitlines())} lines of HTML generated")

In [ ]:
#@title 5.2 · Try it (only after the safety scan)
if not found:
    display(HTML(my_tool))
else:
    print("Ask the model to remove the flagged items (5.3), then run 5.1's scan again.")

In [ ]:
#@title 5.3 · Refine: describe the change, get the full updated file back
CHANGE = "Add a 'Restart quiz' button and show the score as a percentage"  #@param {type:"string"}
my_tool = extract_block(ask(
    f"Here is the current HTML file:\n```html\n{my_tool}\n```\n\nChange request: {CHANGE}\n"
    "Keep all constraints: single file, no external resources, no network requests, no storage, no patient data. "
    "Return the complete updated file in one ```html block.", temperature=0, max_tokens=8000), "html")
found = [f for f in RED_FLAGS if f.lower() in my_tool.lower()]
print("Safety scan:", "no red flags found" if not found else f"REVIEW NEEDED -> {found}")
if not found: display(HTML(my_tool))

In [ ]:
#@title 5.4 · Download your tool
name = "my_radiology_tool.html"  #@param {type:"string"}
with open(name, "w") as fh: fh.write(my_tool)
try:
    from google.colab import files; files.download(name)
except Exception:
    print("Saved as", name)

**Before you use any tool you built:** test it with cases where you know the answer, ask a colleague to try to break it (Show & Tell), and keep patient data out unless your institution has reviewed it.